# A04 — BPE from scratch

Due 5:00 AM Fri 18 September. Follow [minbpe's exercise](https://github.com/karpathy/minbpe/blob/master/exercise.md):
Step 1 `BasicTokenizer`, Step 2 `RegexTokenizer`, Step 3 stretch.
**Steps 1 and 2 are the assignment.** Skip Step 3 and still earn full marks.

Write your code in `bpe/tokenizer.py`, not in this notebook. The notebook is where
you drive it and look at what it did; the tests import the module.


In [ ]:
import sys, importlib
sys.path.insert(0, "..")

import bpe.tokenizer as T
importlib.reload(T)          # re-run this cell after every edit to the module

print([n for n in dir(T) if not n.startswith("_")])


## Before anything else: the regex module

`\p{L}` is a Unicode property escape. The standard library's `re` does not support it,
and the error you get points at the pattern rather than at the import.


In [ ]:
import regex          # NOT `re`
assert regex.compile(T.GPT4_SPLIT_PATTERN), "pattern failed to compile"
print("regex ok")


## Your corpus

Something you actually care about, at least ~1MB: your own code, a book, a Discord
export, scraped lyrics. Your vocabulary is only interesting if your corpus has a
personality. Put it in `data/` (which `.gitignore` already excludes) and point at it.


In [ ]:
from pathlib import Path

text = Path("../data/corpus.txt").read_text(encoding="utf-8")
print(f"{len(text):,} characters, {len(text.encode('utf-8')):,} bytes")


## Step 1 — BasicTokenizer

Train to 512 merges (vocab 768). More is fine; fewer is not enough to see structure.


In [ ]:
importlib.reload(T)
basic = T.BasicTokenizer()
basic.train(text, 768, verbose=True)

print(f"{len(basic.merges)} merges learned")
for n, (pair, idx) in enumerate(basic.merges.items(), 1):
    if n in (1, 10, 100):
        print(f"  merge #{n:>3}: {basic.vocab[idx]!r}")


### Round-trip before you compare anything

A tokenizer that does not round-trip makes every later number meaningless.


In [ ]:
assert basic.decode(basic.encode(text)) == text, "round-trip failed on the corpus"
for s in ["", "a", "h\u00e9llo", "\U0001F642", "    def foo():", "12,345,678"]:
    assert basic.decode(basic.encode(s)) == s, f"round-trip failed on {s!r}"
print("round-trip clean")


## Step 2 — RegexTokenizer

Same BPE, but the text is split before any pair is counted, so no merge is ever
learned across a category boundary.


In [ ]:
importlib.reload(T)
rt = T.RegexTokenizer()
rt.train(text, 768, verbose=True)

assert rt.decode(rt.encode(text)) == text
print(f"{len(rt.merges)} merges")
print(f"basic: {len(basic.encode(text)):,} ids")
print(f"regex: {len(rt.encode(text)):,} ids")


## Compression, against `cl100k_base`

Bytes per token, on **your** corpus. Note before you report it why this comparison
is unfair in your favour.


In [ ]:
import tiktoken
gpt4 = tiktoken.get_encoding("cl100k_base")
nbytes = len(text.encode("utf-8"))

for name, ids in (("yours (regex)", rt.encode(text)), ("cl100k_base", gpt4.encode(text))):
    print(f"{name:<16} {len(ids):>8,} ids   {nbytes / len(ids):.2f} bytes/token")


## Step 3 — stretch, not required

Matching GPT-4 byte-for-byte. Everything above is the assignment; this is here for
the person who wants it.


## Then

```
pytest tests/ -q
```

and write `bpe/DIFF.md`: your vocabulary against `cl100k_base`, with three specific
differences explained.
